# 03 - Evaluation: Length Generalisation

This notebook evaluates trained models on **unseen input lengths** to measure length generalisation.

Key questions:
- Does the Karatsuba-trained model generalise to 16-bit, 32-bit, 64-bit multiplication?
- How does it compare to the school algorithm baseline?
- Where do errors occur (which bits, which recursion levels)?

**Run on Colab with T4 GPU (free tier). Inference is fast for small models.**

In [ ]:
# Cell 1: Load trained models
import os
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.random as jrandom
import equinox as eqx

# --- Repo setup ---
REPO_URL = "https://github.com/YOUR_USERNAME/karatsuba-transformers.git"  # TODO: update
REPO_DIR = "/content/karatsuba-transformers"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

!pip install -q -r {REPO_DIR}/requirements.txt
!pip install -q -e {REPO_DIR}
sys.path.insert(0, REPO_DIR)

# Mount Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_BASE = '/content/drive/MyDrive/karatsuba_checkpoints'

from src.model.looped_transformer import KaratsubaTransformer
from src.training.evaluate import Evaluator
from src.data.dataset import MultiplicationDataset
from src.data.karatsuba_trace import generate_karatsuba_trace

# Load configs
with open(os.path.join(REPO_DIR, 'configs', '8bit_karatsuba.yaml'), 'r') as f:
    config_karatsuba = yaml.safe_load(f)
with open(os.path.join(REPO_DIR, 'configs', '8bit_school_baseline.yaml'), 'r') as f:
    config_school = yaml.safe_load(f)

# Load trained models from checkpoints
print("Loading Karatsuba model...")
karatsuba_ckpt = os.path.join(CHECKPOINT_BASE, '8bit_karatsuba', 'best_model.eqx')
key = jrandom.PRNGKey(42)
model_karatsuba = KaratsubaTransformer(config_karatsuba['model'], key=key)
model_karatsuba = eqx.tree_deserialise_leaves(karatsuba_ckpt, model_karatsuba)
print(f"  Loaded from: {karatsuba_ckpt}")

print("Loading School model...")
school_ckpt = os.path.join(CHECKPOINT_BASE, '8bit_school_baseline', 'best_model.eqx')
model_school = KaratsubaTransformer(config_school['model'], key=key)
model_school = eqx.tree_deserialise_leaves(school_ckpt, model_school)
print(f"  Loaded from: {school_ckpt}")

print("\nModels loaded successfully.")

In [ ]:
# Cell 2: Evaluate at each test length

test_lengths = [8, 16, 32, 64]
n_eval = 1000  # samples per length

# Loop counts for each test length (from config)
karatsuba_loop_counts = config_karatsuba['evaluation']['test_loop_counts']
school_loop_counts = config_school['evaluation']['test_loop_counts']

results_karatsuba = {}
results_school = {}

evaluator_k = Evaluator(model_karatsuba, config_karatsuba)
evaluator_s = Evaluator(model_school, config_school)

for n_bits in test_lengths:
    print(f"\n{'='*50}")
    print(f"Evaluating at {n_bits}-bit x {n_bits}-bit")
    print(f"{'='*50}")

    # Generate test data
    np.random.seed(12345 + n_bits)  # Deterministic test set
    test_x = np.random.randint(0, 2**n_bits, size=n_eval)
    test_y = np.random.randint(0, 2**n_bits, size=n_eval)

    # Evaluate Karatsuba model
    n_loops_k = karatsuba_loop_counts.get(str(n_bits), karatsuba_loop_counts.get(n_bits, 20))
    print(f"  Karatsuba: {n_loops_k} loops")
    res_k = evaluator_k.evaluate(test_x, test_y, n_bits, n_loops=n_loops_k)
    results_karatsuba[n_bits] = res_k
    print(f"    Exact match: {res_k['exact_match']:.4f}")
    print(f"    Per-digit mean: {np.mean(res_k['per_digit_accuracy']):.4f}")

    # Evaluate School model
    n_loops_s = school_loop_counts.get(str(n_bits), school_loop_counts.get(n_bits, 20))
    print(f"  School: {n_loops_s} loops")
    res_s = evaluator_s.evaluate(test_x, test_y, n_bits, n_loops=n_loops_s)
    results_school[n_bits] = res_s
    print(f"    Exact match: {res_s['exact_match']:.4f}")
    print(f"    Per-digit mean: {np.mean(res_s['per_digit_accuracy']):.4f}")

# Summary table
print(f"\n\n{'Length Generalisation Summary':^60}")
print("=" * 60)
print(f"{'Bit Width':>10} {'Karatsuba':>12} {'School':>12} {'Improvement':>14}")
print("-" * 60)
for n_bits in test_lengths:
    k_acc = results_karatsuba[n_bits]['exact_match']
    s_acc = results_school[n_bits]['exact_match']
    improvement = k_acc - s_acc
    marker = " ***" if improvement > 0.1 else ""
    print(f"{n_bits:>10} {k_acc:>12.4f} {s_acc:>12.4f} {improvement:>+13.4f}{marker}")

In [ ]:
# Cell 3: Per-digit accuracy analysis
# Which output bits are hardest? Do errors concentrate in MSBs (carry propagation)?

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, n_bits in enumerate(test_lengths):
    ax = axes[idx]
    n_output_bits = 2 * n_bits

    pda_k = results_karatsuba[n_bits]['per_digit_accuracy']
    pda_s = results_school[n_bits]['per_digit_accuracy']

    bit_positions = np.arange(len(pda_k))
    width = 0.35

    ax.bar(bit_positions - width/2, pda_k, width, label='Karatsuba', color='steelblue', alpha=0.8)
    ax.bar(bit_positions + width/2, pda_s, width, label='School', color='salmon', alpha=0.8)

    ax.set_xlabel('Output Bit Position (0=LSB)', fontsize=10)
    ax.set_ylabel('Accuracy', fontsize=10)
    ax.set_title(f'{n_bits}-bit x {n_bits}-bit', fontsize=12)
    ax.set_ylim([0, 1.05])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2, axis='y')

    # Annotate exact match
    k_em = results_karatsuba[n_bits]['exact_match']
    s_em = results_school[n_bits]['exact_match']
    ax.text(0.02, 0.98, f'Exact: K={k_em:.3f}, S={s_em:.3f}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Per-Bit Accuracy Across Test Lengths', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_BASE, 'per_digit_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()

# Analysis: where do errors concentrate?
print("\nError Analysis:")
for n_bits in test_lengths:
    pda_k = results_karatsuba[n_bits]['per_digit_accuracy']
    if len(pda_k) > 0:
        lsb_acc = np.mean(pda_k[:len(pda_k)//2])
        msb_acc = np.mean(pda_k[len(pda_k)//2:])
        print(f"  {n_bits}-bit Karatsuba: LSB half avg={lsb_acc:.4f}, MSB half avg={msb_acc:.4f}")

In [ ]:
# Cell 4: Per-recursion-level accuracy (Karatsuba only)
# Does the model handle the first recursion level correctly but fail deeper?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-recursion-level accuracy for each test length
ax = axes[0]
for n_bits in test_lengths:
    if 'per_level_accuracy' in results_karatsuba[n_bits]:
        level_accs = results_karatsuba[n_bits]['per_level_accuracy']
        levels = range(len(level_accs))
        ax.plot(levels, level_accs, 'o-', label=f'{n_bits}-bit', markersize=8, linewidth=2)

ax.set_xlabel('Recursion Level (0=top, deeper=smaller sub-problems)', fontsize=11)
ax.set_ylabel('Accuracy at Level', fontsize=11)
ax.set_title('Karatsuba: Accuracy by Recursion Level', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

# Accuracy vs test length (main generalisation plot)
ax = axes[1]
k_accs = [results_karatsuba[b]['exact_match'] for b in test_lengths]
s_accs = [results_school[b]['exact_match'] for b in test_lengths]

ax.plot(test_lengths, k_accs, 'bs-', label='Karatsuba', linewidth=2, markersize=10)
ax.plot(test_lengths, s_accs, 'ro-', label='School', linewidth=2, markersize=10)

# Reference lines
ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='90% threshold')
ax.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='50% threshold')
ax.axvline(x=8, color='gray', linestyle=':', alpha=0.5, label='Training length')

ax.set_xlabel('Test Bit Width', fontsize=12)
ax.set_ylabel('Exact Match Accuracy', fontsize=12)
ax.set_title('Length Generalisation', fontsize=14)
ax.set_xscale('log', base=2)
ax.set_xticks(test_lengths)
ax.set_xticklabels([str(t) for t in test_lengths])
ax.legend(fontsize=10, loc='lower left')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_BASE, 'length_generalisation.png'), dpi=150, bbox_inches='tight')
plt.show()

# Determine generalisation factor
for threshold in [0.9, 0.7, 0.5]:
    max_gen_k = max([b for b in test_lengths if results_karatsuba[b]['exact_match'] >= threshold], default=0)
    max_gen_s = max([b for b in test_lengths if results_school[b]['exact_match'] >= threshold], default=0)
    if max_gen_k > 0:
        print(f"At {threshold*100:.0f}% accuracy threshold:")
        print(f"  Karatsuba generalises to {max_gen_k}-bit ({max_gen_k/8:.0f}x training length)")
        print(f"  School generalises to {max_gen_s}-bit ({max_gen_s/8:.0f}x training length)")

In [ ]:
# Cell 5: Generate comparison plots for paper/report

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Main generalisation curve ---
ax = axes[0]
k_accs = [results_karatsuba[b]['exact_match'] for b in test_lengths]
s_accs = [results_school[b]['exact_match'] for b in test_lengths]

ax.plot(test_lengths, k_accs, 'bs-', label='Karatsuba (ours)', linewidth=2.5, markersize=10, zorder=5)
ax.plot(test_lengths, s_accs, 'ro--', label='School baseline', linewidth=2, markersize=8)
ax.fill_between(test_lengths, s_accs, k_accs, alpha=0.1, color='blue')

ax.axvline(x=8, color='gray', linestyle=':', alpha=0.7, linewidth=1.5)
ax.annotate('Training\nlength', xy=(8, 0.05), fontsize=9, ha='center', color='gray')

ax.set_xlabel('Test Bit Width', fontsize=12)
ax.set_ylabel('Exact Match Accuracy', fontsize=12)
ax.set_title('Length Generalisation: Karatsuba vs School', fontsize=13)
ax.set_xscale('log', base=2)
ax.set_xticks(test_lengths)
ax.set_xticklabels([str(t) for t in test_lengths])
ax.legend(fontsize=11)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.2)

# --- Plot 2: Accuracy vs number of loops ---
ax = axes[1]
loop_counts_to_test = [4, 6, 8, 12, 16, 20, 24]
test_bit = 32  # Test at 32-bit to show loop sensitivity

# This would be populated by running evaluator_k.evaluate with different n_loops
# Placeholder structure:
ax.set_xlabel('Number of Loops', fontsize=12)
ax.set_ylabel('Exact Match Accuracy (32-bit)', fontsize=12)
ax.set_title('Accuracy vs Loop Count at 32-bit', fontsize=13)
ax.text(0.5, 0.5, 'Run evaluations with\nvarying loop counts\nto populate this plot',
        ha='center', va='center', transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.grid(True, alpha=0.2)

# Evaluate with varying loop counts
print("Evaluating Karatsuba at 32-bit with varying loop counts...")
np.random.seed(12345 + test_bit)
test_x_32 = np.random.randint(0, 2**test_bit, size=500)
test_y_32 = np.random.randint(0, 2**test_bit, size=500)

loop_accs = []
for n_loops in loop_counts_to_test:
    res = evaluator_k.evaluate(test_x_32, test_y_32, test_bit, n_loops=n_loops)
    loop_accs.append(res['exact_match'])
    print(f"  {n_loops} loops: {res['exact_match']:.4f}")

ax.clear()
ax.plot(loop_counts_to_test, loop_accs, 'gs-', linewidth=2, markersize=8)
ax.set_xlabel('Number of Loops', fontsize=12)
ax.set_ylabel('Exact Match Accuracy', fontsize=12)
ax.set_title(f'Accuracy vs Loop Count ({test_bit}-bit)', fontsize=13)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.2)

# --- Plot 3: Generalisation ratio ---
ax = axes[2]
gen_ratios = [b / 8 for b in test_lengths]  # ratio to training length
ax.plot(gen_ratios, k_accs, 'bs-', label='Karatsuba', linewidth=2.5, markersize=10)
ax.plot(gen_ratios, s_accs, 'ro--', label='School', linewidth=2, markersize=8)

ax.set_xlabel('Generalisation Ratio (test/train length)', fontsize=12)
ax.set_ylabel('Exact Match Accuracy', fontsize=12)
ax.set_title('Accuracy vs Generalisation Ratio', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim([0, 1.05])
ax.set_xticks(gen_ratios)
ax.set_xticklabels([f'{r:.0f}x' for r in gen_ratios])
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_BASE, 'paper_figure_generalisation.png'), dpi=300, bbox_inches='tight')
plt.show()

print("\nFigures saved to Google Drive.")